# Qwen3-4B — Fine-tune ViNumQA with synthesized reasoning traces (Unsloth)

Goal: fine-tune `unsloth/Qwen3-4B` on the ViNumQA (VLSP-2025 numerical reasoning) train split so the
model learns to emit a `<think>...</think>` reasoning trace before the final FinQA-style computation
program, instead of jumping straight to the bare program string like the `sft/` baseline notebook does.

The dataset itself has **no** natural-language chain-of-thought — only `(context, question) -> program,
exe_ans)`. So this notebook **synthesizes** a grounded reasoning trace by verbalizing each step of the
gold program (looking up table rows by label, resolving `#N` references, doing the actual arithmetic)
and wraps it in Qwen3's `<think>` block. The visible final answer stays the bare program string, so it's
directly comparable with the 0-shot/1-shot/SFT notebooks using the same Program Accuracy / Execution
Accuracy metrics.

Run on Kaggle (T4/P100 free tier). Loosely follows the structure of
`Qwen3_(14B)_Reasoning_Conversational.ipynb` in this same folder, adapted for:
- Qwen3-**4B** instead of 14B (fits comfortably on a single T4)
- ViNumQA instead of OpenMathReasoning / FineTome
- reasoning traces synthesized from gold programs instead of pulled from an existing dataset

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups (incl. Kaggle)
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install -q tabulate
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import os
from huggingface_hub import login

# Optional: only needed if you plan to push the adapter/merged model to the Hub.
# On Kaggle: Add-ons > Secrets > add "HF_TOKEN", then attach it to this notebook.
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    login(HF_TOKEN)
else:
    print("No HF_TOKEN found -- skipping login (fine unless you want to push_to_hub).")

### Load base model + LoRA adapters

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 4096  # ViNumQA contexts (pre_text + table + post_text) can be long

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B",
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

<a name="Data"></a>
### Data prep

Load the official 3-way ViNumQA split (same source/path as `sft/vsf-fintech-vinumqa-sft-qwen3-4b.ipynb`)
and reuse the same context formatting (pre_text / markdown table / post_text) and system prompt so results
stay comparable across the 0-shot, 1-shot, SFT and this reasoning-SFT notebook.

In [ ]:
import pandas as pd
from tabulate import tabulate

train_df = pd.read_json('/kaggle/input/datasets/ntphuc149x2/vlsp2025-vinumqa/train.json')
valid_df = pd.read_json('/kaggle/input/datasets/ntphuc149x2/vlsp2025-vinumqa/valid.json')
test_df = pd.read_json('/kaggle/input/datasets/ntphuc149x2/vlsp2025-vinumqa/test.json')
print(f"train={len(train_df)}, valid={len(valid_df)}, test={len(test_df)}")

In [ ]:
def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

def processing_input_question(sample):
    return sample["qa"]["question"]

def processing_program_content(sample):
    return sample["qa"]["program"]

def processing_answer_content(sample):
    return sample["qa"]["exe_ans"]

def process_split(df):
    df = df.copy()
    df["pre_text_processed"] = df.apply(formatting_pre_text, axis=1)
    df["post_text_processed"] = df.apply(formatting_post_text, axis=1)
    df["table_processed"] = df.apply(formatting_table, axis=1)
    # keep the raw table (list-of-lists) too -- needed to resolve row-label
    # arguments like table_average("OP Margin (%)", none) into real numbers
    # when synthesizing the reasoning trace below.
    df["table_raw"] = df["table"]
    df["input_question"] = df.apply(processing_input_question, axis=1)
    df["program_processed"] = df.apply(processing_program_content, axis=1)
    df["answer_processed"] = df.apply(processing_answer_content, axis=1)
    df = df[["pre_text_processed", "table_processed", "table_raw", "post_text_processed",
             "input_question", "program_processed", "answer_processed"]]
    df.columns = ["pre_text", "table", "table_raw", "post_text", "question", "program", "answer"]
    return df

train_df = process_split(train_df)
valid_df = process_split(valid_df)
test_df = process_split(test_df)
test_df["generated_program"] = ""

train_df.sample(n=3)

In [ ]:
SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.

### LIST OF 10 VALID OPERATORS:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(val1, val2, val3, ...) -> sum of the values
8. table_average(val1, val2, val3, ...) -> arithmetic mean of the values
9. table_max(val1, val2, val3, ...) -> maximum of the values
10. table_min(val1, val2, val3, ...) -> minimum of the values

### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- Do not use square brackets `[]` inside table-type functions (e.g. write table_max(1, 2, 3), not table_max([1, 2, 3])).
- Think step by step inside <think>...</think>, then output only the final program string after </think>.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use \'none\'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}

[TABLE]
{table}

[TEXT AFTER TABLE]
{post_text}

### QUESTION:
{question}

### PROGRAM:"""

# Same prompt format as the 0-shot/1-shot/sft notebooks (only the "Do not
# perform mental calculations" rule is swapped for a "think step by step"
# rule, since this notebook trains the model to *do* the reasoning inline).

### Synthesize reasoning traces from gold programs

ViNumQA gives us the final `program` string (e.g. `subtract(1781, 1675), divide(#0, 1675)`) and the
executed answer, but no natural-language explanation. `build_reasoning_trace` verbalizes each step in
Vietnamese, actually executing the arithmetic (and resolving `table_*` row-label arguments against the
raw table) so the numbers quoted in the trace are grounded in real values whenever they can be resolved.
If a step can't be resolved (unusual argument format, etc.) it still describes the operation in words and
moves on -- the trace is best-effort narration, while the trainable target (the program string / exe_ans)
always comes straight from the gold data, so training supervision stays correct either way.

In [ ]:
import re

_NUM_RE = re.compile(r"^-?\d+(\.\d+)?$")
_REF_RE = re.compile(r"^#(\d+)$")
_BINARY_OPS = {"add", "subtract", "multiply", "divide", "exp", "greater"}
_TABLE_OPS = {"table_sum", "table_average", "table_max", "table_min"}
_VALID_OPERATORS = _BINARY_OPS | _TABLE_OPS

_OP_VERBS = {
    "add": "cộng {a} với {b}",
    "subtract": "lấy {a} trừ đi {b}",
    "multiply": "nhân {a} với {b}",
    "divide": "lấy {a} chia cho {b}",
    "exp": "tính {a} lũy thừa {b}",
    "greater": "so sánh {a} với {b}",
}


def _parse_num(raw):
    cleaned = str(raw).replace(",", "").replace("$", "").strip()
    is_pct = cleaned.endswith("%")
    if is_pct:
        cleaned = cleaned[:-1].strip()
    if not _NUM_RE.match(cleaned):
        return None
    val = float(cleaned)
    return val / 100.0 if is_pct else val


def _split_top_level(s, sep=","):
    parts, depth, current = [], 0, []
    for ch in s:
        if ch == "(":
            depth += 1; current.append(ch)
        elif ch == ")":
            depth -= 1; current.append(ch)
        elif ch == sep and depth == 0:
            parts.append("".join(current)); current = []
        else:
            current.append(ch)
    if current:
        parts.append("".join(current))
    return [p.strip() for p in parts if p.strip() != ""]


def _parse_program_steps(program_str):
    """Best-effort parse; returns [] instead of raising on malformed input.

    Unlike the strict FinQA-style parser used for scoring (below), this scans
    for balanced parentheses rather than `[^()]*`, because ViNumQA table_*
    arguments are often table row *labels* that themselves contain parens,
    e.g. `table_max(EPS (VND), none)` -- a non-nesting-aware regex would fail
    to parse ~9% of gold programs and silently drop their reasoning steps.
    """
    text = str(program_str).strip().rstrip(",").strip()
    if not text:
        return []
    steps = []
    ident_re = re.compile(r"[a-zA-Z_]+")
    pos, n = 0, len(text)
    while pos < n:
        m = ident_re.match(text, pos)
        if not m:
            pos += 1
            continue
        op = m.group(0)
        after = m.end()
        if after < n and text[after] == "(":
            depth, j = 0, after
            while j < n:
                if text[j] == "(":
                    depth += 1
                elif text[j] == ")":
                    depth -= 1
                    if depth == 0:
                        break
                j += 1
            if depth == 0 and j < n:
                args_str = text[after + 1:j]
                if op in _VALID_OPERATORS:
                    steps.append((op, _split_top_level(args_str, sep=",")))
                pos = j + 1
                continue
        pos = after
    return steps


def _fmt(x):
    if x is None:
        return "?"
    x = float(x)
    return str(int(x)) if x.is_integer() else f"{x:g}"


def _lookup_table_row(table, label):
    if not table or len(table) < 2:
        return None
    for row in table[1:]:
        if row and str(row[0]).strip().lower() == str(label).strip().lower():
            return row[1:]
    return None


def build_reasoning_trace(question, table, program_str, exe_ans):
    steps = _parse_program_steps(program_str)
    lines = [
        f"Câu hỏi cần trả lời: {str(question).strip()}",
        "Ta phân tích ngữ cảnh và bảng số liệu, thực hiện tính toán theo từng bước của chương trình:",
    ]
    results = []
    for step_no, (op, raw_args) in enumerate(steps, start=1):
        resolved_groups, display_args = [], []
        for a in raw_args:
            a = a.strip()
            ref_m = _REF_RE.match(a)
            if ref_m:
                idx = int(ref_m.group(1))
                val = results[idx] if idx < len(results) else None
                resolved_groups.append([val] if val is not None else [])
                display_args.append(f"kết quả bước {idx + 1}" + (f" ({_fmt(val)})" if val is not None else ""))
            elif a.lower() == "none":
                resolved_groups.append([])
                display_args.append(None)
            else:
                num = _parse_num(a)
                if num is not None:
                    resolved_groups.append([num])
                    display_args.append(_fmt(num))
                else:
                    row_vals = _lookup_table_row(table, a)
                    nums = [v for v in (_parse_num(x) for x in (row_vals or [])) if v is not None]
                    resolved_groups.append(nums)
                    display_args.append(f'các giá trị ở dòng "{a}" trong bảng')

        flat = [v for group in resolved_groups for v in group]
        r = None
        try:
            if op in _BINARY_OPS:
                a_vals = resolved_groups[0] if len(resolved_groups) > 0 else []
                b_vals = resolved_groups[1] if len(resolved_groups) > 1 else []
                a_val = a_vals[0] if a_vals else None
                b_val = b_vals[0] if b_vals else None
                if a_val is not None and b_val is not None:
                    if op == "add": r = a_val + b_val
                    elif op == "subtract": r = a_val - b_val
                    elif op == "multiply": r = a_val * b_val
                    elif op == "divide": r = (a_val / b_val) if b_val != 0 else None
                    elif op == "exp": r = a_val ** b_val
                    elif op == "greater": r = 1.0 if a_val > b_val else 0.0
            elif flat:
                if op == "table_sum": r = sum(flat)
                elif op == "table_average": r = sum(flat) / len(flat)
                elif op == "table_max": r = max(flat)
                elif op == "table_min": r = min(flat)
        except Exception:
            r = None
        results.append(r)

        if op in _BINARY_OPS:
            a_disp = display_args[0] if len(display_args) > 0 and display_args[0] else "?"
            b_disp = display_args[1] if len(display_args) > 1 and display_args[1] else "?"
            text = f"Bước {step_no}: {_OP_VERBS[op].format(a=a_disp, b=b_disp)}"
        else:
            args_str = " và ".join(d for d in display_args if d) or "các giá trị liên quan"
            verb = {"table_sum": "tính tổng", "table_average": "tính trung bình",
                    "table_max": "lấy giá trị lớn nhất trong", "table_min": "lấy giá trị nhỏ nhất trong"}[op]
            text = f"Bước {step_no}: {verb} {args_str}"
        text += f" → {_fmt(r)}." if r is not None else "."
        lines.append(text)

    lines.append(f"Vậy kết quả cuối cùng của phép tính là {exe_ans}.")
    return "\n".join(lines)


# Sanity check on one training example
_sample = train_df.iloc[0]
print(build_reasoning_trace(_sample["question"], _sample["table_raw"], _sample["program"], _sample["answer"]))
print("---")
print(_sample["program"])

### Build the conversational (reasoning) dataset

In [ ]:
def build_conversation(row):
    user_msg = USER_MESSAGE_FRAME.format(
        pre_text=row["pre_text"], table=row["table"],
        post_text=row["post_text"], question=row["question"],
    )
    trace = build_reasoning_trace(row["question"], row["table_raw"], row["program"], row["answer"])
    assistant_msg = f"<think>\n{trace}\n</think>\n\n{str(row['program']).strip()}"
    return [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]


def make_text_dataset(df):
    conversations = [build_conversation(row) for _, row in df.iterrows()]
    texts = tokenizer.apply_chat_template(conversations, tokenize=False)
    return texts


train_texts = make_text_dataset(train_df)
valid_texts = make_text_dataset(valid_df)
print(len(train_texts), len(valid_texts))
print(train_texts[0][:1500])

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"text": train_texts}).shuffle(seed=3407)
valid_dataset = Dataset.from_dict({"text": valid_texts})
train_dataset, valid_dataset

<a name="Train"></a>
### Train the model

We mask the loss to only the assistant turn (`train_on_responses_only`) so the model isn't penalized for
"predicting" the system prompt / context / question -- standard practice, and important here since the
context block can be much longer than the reasoning + program we actually want it to learn to produce.

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "/kaggle/working/qwen3-4b-vinumqa-reasoning"

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = valid_dataset,
    args = SFTConfig(
        output_dir = OUTPUT_DIR,
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        per_device_train_batch_size = 2,
        per_device_eval_batch_size = 2,
        gradient_accumulation_steps = 8,   # effective batch size = 16
        warmup_ratio = 0.03,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 20,
        eval_strategy = "epoch",
        save_strategy = "epoch",
        save_total_limit = 2,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 3407,
        report_to = "none",
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

In [ ]:
# @title Show current memory stats
import torch
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

Train the model. To resume a run, set `trainer.train(resume_from_checkpoint = True)`.

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference

Recommended Qwen3 thinking-mode sampling: `temperature=0.6, top_p=0.95, top_k=20`.

In [ ]:
FastLanguageModel.for_inference(model)

_row = test_df.iloc[0]
_prompt = USER_MESSAGE_FRAME.format(
    pre_text=_row["pre_text"], table=_row["table"], post_text=_row["post_text"], question=_row["question"],
)
messages = [
    {"role": "system", "content": SYSTEM_MESSAGE},
    {"role": "user", "content": _prompt},
]
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=True,
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    max_new_tokens=1536,
    temperature=0.6, top_p=0.95, top_k=20,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)
print("\nGold program:", _row["program"], "| Gold answer:", _row["answer"])

<a name="Save"></a>
### Saving

Saves LoRA adapters locally. Uncomment the `push_to_hub` lines (and set `HF_USERNAME`) to also upload.
See the merged-16bit / GGUF cells below for other export options (same as the reference notebook).

In [ ]:
ADAPTER_DIR = "/kaggle/working/qwen3-4b-vinumqa-reasoning-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Saved LoRA adapter to {ADAPTER_DIR}")

# HF_USERNAME = "your_hf_username"
# model.push_to_hub(f"{HF_USERNAME}/qwen3-4b-vinumqa-reasoning-adapter", token=HF_TOKEN)
# tokenizer.push_to_hub(f"{HF_USERNAME}/qwen3-4b-vinumqa-reasoning-adapter", token=HF_TOKEN)

In [ ]:
# Merge to 16bit / 4bit, or export GGUF for llama.cpp -- disabled by default.
if False:
    model.save_pretrained_merged("qwen3-4b-vinumqa-reasoning-16bit", tokenizer, save_method="merged_16bit")
if False:
    model.save_pretrained_merged("qwen3-4b-vinumqa-reasoning-4bit", tokenizer, save_method="merged_4bit")
if False:
    model.save_pretrained_gguf("qwen3-4b-vinumqa-reasoning", tokenizer, quantization_method="q4_k_m")

<a name="Eval"></a>
### PA / EA on the ViNumQA test set

Quick check of Program Accuracy (PA) / Execution Accuracy (EA) after fine-tuning, using the same parser
as the 0-shot/1-shot/SFT notebooks so numbers are comparable. The model emits `<think>...</think>` first;
we strip everything up to and including the `</think>` token (id `151668` for Qwen3) before extracting
the program string. No checkpointing / result-file export here -- just the PA/EA numbers.

In [ ]:
"""
Parser + evaluation utilities for FinQA/VLSP-2025 style computation programs.
(Same implementation as the 0-shot/1-shot/sft notebooks.)
"""

import re
from typing import List, Tuple, Union, Optional

VALID_OPERATORS = {
    "add", "subtract", "multiply", "divide", "exp", "greater",
    "table_sum", "table_average", "table_max", "table_min",
}
_BINARY_OPS = {"add", "subtract", "multiply", "divide", "exp", "greater"}
_TABLE_OPS = {"table_sum", "table_average", "table_max", "table_min"}
_NUM_RE = re.compile(r"^-?\d+(\.\d+)?$")
_REF_RE = re.compile(r"^#(\d+)$")


def _parse_numeric_literal(raw: str) -> float:
    cleaned = raw.replace(",", "").replace("$", "").strip()
    is_percent = cleaned.endswith("%")
    if is_percent:
        cleaned = cleaned[:-1].strip()
    if not _NUM_RE.match(cleaned):
        raise ValueError(f"Cannot parse numeric argument: '{raw}'")
    value = float(cleaned)
    return value / 100.0 if is_percent else value


def extract_program(raw_text: str) -> str:
    text = raw_text.strip()
    text = re.sub(r"```[a-zA-Z]*", "", text)
    text = text.replace("```", "").strip()
    op_names = "|".join(sorted(VALID_OPERATORS, key=len, reverse=True))
    pattern = rf"\b(?:{op_names})\([^()]*\)"
    matches = re.findall(pattern, text)
    if matches:
        return ", ".join(m.strip() for m in matches)
    return text


def _split_top_level(s: str, sep: str = ",") -> List[str]:
    parts, depth, current = [], 0, []
    for ch in s:
        if ch == "(":
            depth += 1; current.append(ch)
        elif ch == ")":
            depth -= 1; current.append(ch)
        elif ch == sep and depth == 0:
            parts.append("".join(current)); current = []
        else:
            current.append(ch)
    if current:
        parts.append("".join(current))
    return [p.strip() for p in parts if p.strip() != ""]


def parse_program(program_str: str) -> List[Tuple[str, List[str]]]:
    program_str = program_str.strip().rstrip(",").strip()
    if not program_str:
        raise ValueError("Empty program string.")
    steps: List[Tuple[str, List[str]]] = []
    call_pattern = re.compile(r"\s*([a-zA-Z_]+)\(([^()]*)\)\s*")
    pos = 0
    text = program_str
    while pos < len(text):
        m = call_pattern.match(text, pos)
        if not m:
            raise ValueError(f"Malformed program near: '{text[pos:pos+30]}...'")
        op = m.group(1).strip()
        if op not in VALID_OPERATORS:
            raise ValueError(f"Unknown operator: '{op}'")
        args = _split_top_level(m.group(2), sep=",")
        steps.append((op, args))
        pos = m.end()
        if pos < len(text) and text[pos] == ",":
            pos += 1
    if not steps:
        raise ValueError("No valid steps parsed.")
    return steps


def _resolve_arg(arg: str, results: List[float]) -> Optional[float]:
    arg = arg.strip()
    ref_match = _REF_RE.match(arg)
    if ref_match:
        idx = int(ref_match.group(1))
        if idx >= len(results):
            raise ValueError(f"Reference #{idx} used before step {idx} was computed.")
        return results[idx]
    if arg.lower() == "none":
        return None
    return _parse_numeric_literal(arg)


def execute_program(program_str: str) -> Tuple[float, List[float]]:
    steps = parse_program(program_str)
    results: List[float] = []
    for op, raw_args in steps:
        vals = [_resolve_arg(a, results) for a in raw_args]
        if op in _BINARY_OPS:
            if len(vals) != 2:
                raise ValueError(f"Operator '{op}' expects 2 args, got {len(vals)}.")
            a, b = vals
            if a is None or b is None:
                raise ValueError(f"Operator '{op}' received a 'none' operand.")
            if op == "add": r = a + b
            elif op == "subtract": r = a - b
            elif op == "multiply": r = a * b
            elif op == "divide":
                if b == 0:
                    raise ValueError("Division by zero.")
                r = a / b
            elif op == "exp": r = a ** b
            elif op == "greater": r = 1.0 if a > b else 0.0
        elif op in _TABLE_OPS:
            if len(vals) < 1:
                raise ValueError(f"Operator '{op}' requires at least 1 arg.")
            if any(v is None for v in vals):
                raise ValueError(f"Operator '{op}' received a 'none' operand.")
            if op == "table_sum": r = sum(vals)
            elif op == "table_average": r = sum(vals) / len(vals)
            elif op == "table_max": r = max(vals)
            elif op == "table_min": r = min(vals)
        else:
            raise ValueError(f"Unknown operator: '{op}'")
        results.append(r)
    return results[-1], results


def _normalize_program(program_str: str) -> List[Tuple[str, Tuple[str, ...]]]:
    steps = parse_program(program_str)
    normalized = []
    for op, args in steps:
        norm_args = []
        for a in args:
            a = a.strip()
            if _REF_RE.match(a) or a.lower() == "none":
                norm_args.append(a.lower())
            else:
                try:
                    norm_args.append(f"{round(_parse_numeric_literal(a), 6)}")
                except ValueError:
                    norm_args.append(a)
        normalized.append((op, tuple(norm_args)))
    return normalized


def compute_program_accuracy(generated_program, gold_program, alternative_gold_programs=None) -> float:
    try:
        gen_norm = _normalize_program(generated_program)
    except ValueError:
        return 0.0
    gold_candidates = [gold_program] + (alternative_gold_programs or [])
    for gold in gold_candidates:
        try:
            gold_norm = _normalize_program(gold)
        except ValueError:
            continue
        if gen_norm == gold_norm:
            return 1.0
    return 0.0


def compute_execution_accuracy(generated_program, gold_answer, rel_tol: float = 1e-3, abs_tol: float = 1e-4) -> float:
    try:
        predicted, _ = execute_program(generated_program)
    except (ValueError, ZeroDivisionError, OverflowError):
        return 0.0
    try:
        gold = _parse_numeric_literal(str(gold_answer))
    except ValueError:
        return 0.0
    if abs(predicted - gold) <= max(abs_tol, rel_tol * abs(gold)):
        return 1.0
    return 0.0


def evaluate_dataframe(df, generated_col="generated_program", gold_program_col="program",
                        gold_answer_col="answer", extract_first=True):
    df = df.copy()
    pa_scores, ea_scores = [], []
    for _, row in df.iterrows():
        raw_generated = row[generated_col]
        generated = extract_program(raw_generated) if extract_first else raw_generated
        pa = compute_program_accuracy(generated, row[gold_program_col])
        ea = compute_execution_accuracy(generated, row[gold_answer_col])
        pa_scores.append(pa)
        ea_scores.append(ea)
    df["pa_score"] = pa_scores
    df["ea_score"] = ea_scores
    summary = {
        "program_accuracy": sum(pa_scores) / len(pa_scores) if pa_scores else 0.0,
        "execution_accuracy": sum(ea_scores) / len(ea_scores) if ea_scores else 0.0,
    }
    return df, summary

In [ ]:
import gc
from tqdm import tqdm

QWEN3_THINK_END_TOKEN_ID = 151668  # "</think>"

for df_index in tqdm(test_df.index, desc="Generating program..."):
    row = test_df.loc[df_index]
    prompt = USER_MESSAGE_FRAME.format(
        pre_text=row["pre_text"], table=row["table"], post_text=row["post_text"], question=row["question"],
    )
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs, max_new_tokens=1536,
            temperature=0.6, top_p=0.95, top_k=20,
        )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

    try:
        idx = len(output_ids) - output_ids[::-1].index(QWEN3_THINK_END_TOKEN_ID)
    except ValueError:
        idx = 0

    output = tokenizer.decode(output_ids[idx:], skip_special_tokens=True)
    test_df.at[df_index, "generated_program"] = output.strip()

    gc.collect()
    torch.cuda.empty_cache()

print(f"Done. {(test_df['generated_program'] != '').sum()} / {len(test_df)} samples generated.")

In [ ]:
df_scored, summary = evaluate_dataframe(test_df)
print(summary)  # {\'program_accuracy\': ..., \'execution_accuracy\': ...}